# Instrument-Agnostic Automatic Music Transcription (Colab Inference)

This notebook runs inference with the [instrument-agnostic-amt fork](https://github.com/ntamotsu/fork-instrument-agnostic-amt).
Dependencies are installed from the repository's `uv.lock`, and the pre-trained model is downloaded automatically from Hugging Face.

## 1. Setup Environment

In [ ]:
# @title Install dependencies
!git clone https://github.com/ntamotsu/fork-instrument-agnostic-amt.git
%cd fork-instrument-agnostic-amt
!pip install -q "uv==0.8.17"
# Install the locked dependencies (uv.lock) into Colab's system Python.
# The first run downloads the PyTorch 2.13 CUDA wheels and takes a few minutes.
!uv export --frozen --no-dev --extra stem --output-file colab-requirements.txt
# On Linux the lockfile pins PyTorch's cu130 wheels, which live on the extra
# index below (uv export does not embed index URLs in the requirements file).
!uv pip install --system --extra-index-url https://download.pytorch.org/whl/cu130 -r colab-requirements.txt

## 2. Prepare Audio

You can either upload a file or download one from a URL (e.g., YouTube).

In [ ]:
# @title Upload audio file
from google.colab import files
import os

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {audio_path}")

## 3. Stem Separation -> Transcribe Each Stem -> Instrument Refinement -> Merge -> Velocity Prediction -> Beat/Chord/Key Prediction

This section builds on the `batch_process_unlabeled.py` flow inside Colab.
It separates the uploaded song into stems, transcribes each stem, optionally relabels the instrument of every note with the instrument refinement model, merges the per-stem MIDI files into one result, predicts per-note velocity dynamics using separated stem audio, and optionally predicts beat, chord, and key information using the beat_chord model (`best_beat_chord_key.pth`).
Drum stems use the dedicated experimental `drums` model, bass stems use the `bass_v2` model, and guitar stems use the `guitar_v1_5` model.
Instrument refinement (`REFINE_INSTRUMENTS`) listens to each separated stem again and reassigns the instrument class of its notes, so a stem whose notes were labeled with the wrong instrument can be corrected before merging. Drum and vocal stems are always skipped: drums have no non-drum candidate classes, and separating `melody` from `vocal_harmony` is a matter of musical role rather than timbre, which this model cannot judge.

`DEVICE` defaults to `auto`, which picks CUDA → MPS → CPU; on Colab GPU runtimes this selects CUDA. `AMP` (with `AMP_DTYPE`) enables opt-in mixed-precision inference. `COMPILE_MODEL` (with `COMPILE_MODE`) enables opt-in `torch.compile` of the core AMT forward only, and the independent `COMPILE_VELOCITY` opt-in compiles the velocity forward — fixed-length full windows run compiled, while the trailing partial window automatically falls back to eager. The first compiled window includes the compilation time.


In [ ]:
# @title Prepare stem-separated transcription helpers
from infer_stem import run_stem_separated_transcription


In [ ]:
# @title Run stem-separated transcription
OUTPUT_ROOT = "colab_outputs"  # @param {type:"string"}
WINDOW_BATCH_SIZE = 4  # @param {type:"integer"}
MAX_MIDI_MELODIC_INSTRUMENTS = 15  # @param {type:"integer"}
TRANSCRIBE_DRUM_STEMS = True  # @param {type:"boolean"}
REFINE_INSTRUMENTS = False  # @param {type:"boolean"}
REFINEMENT_CHECKPOINT = ""  # @param {type:"string"}
REFINEMENT_MODE = "cluster"  # @param ["cluster", "single"]
PREDICT_VELOCITY = True  # @param {type:"boolean"}
PREDICT_BEAT_CHORD = False  # @param {type:"boolean"}
CLEANUP_SEPARATED_STEMS = False  # @param {type:"boolean"}
MERGE_ONSET_MS = 50.0  # @param {type:"number"}
DEVICE = "auto"  # @param ["auto", "cuda", "mps", "cpu"]
AMP = False  # @param {type:"boolean"}
AMP_DTYPE = "default"  # @param ["default", "fp16", "bf16"]
COMPILE_MODEL = False  # @param {type:"boolean"}
COMPILE_VELOCITY = False  # @param {type:"boolean"}
COMPILE_MODE = "default"  # @param ["default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs"]

if "audio_path" not in globals():
    raise RuntimeError("Please upload an audio file first.")

stem_pipeline_result = run_stem_separated_transcription(
    audio_path,
    checkpoint_path=None,
    output_root=OUTPUT_ROOT,
    window_batch_size=WINDOW_BATCH_SIZE,
    max_midi_melodic_instruments=MAX_MIDI_MELODIC_INSTRUMENTS,
    transcribe_drum_stems=TRANSCRIBE_DRUM_STEMS,
    refine_instruments=REFINE_INSTRUMENTS,
    refinement_checkpoint_path=REFINEMENT_CHECKPOINT or None,
    refinement_mode=REFINEMENT_MODE,
    predict_velocity=PREDICT_VELOCITY,
    predict_beat_chord=PREDICT_BEAT_CHORD,
    cleanup_separated_stems=CLEANUP_SEPARATED_STEMS,
    merge_onset_ms=MERGE_ONSET_MS,
    device=DEVICE,
    amp=AMP,
    amp_dtype=None if AMP_DTYPE == "default" else AMP_DTYPE,
    compile_model=COMPILE_MODEL,
    compile_velocity=COMPILE_VELOCITY,
    compile_mode=COMPILE_MODE,
)
stem_pipeline_result


In [ ]:
# @title Download stem-separated results
from google.colab import files
from pathlib import Path
import shutil

if "stem_pipeline_result" not in globals():
    print("Run the stem-separated transcription cell first.")
else:
    merged_midi_path = Path(stem_pipeline_result["merged_midi_path"])
    stem_midi_dir = Path(stem_pipeline_result["stem_midi_dir"])
    zip_base = stem_midi_dir.parent / f"{stem_midi_dir.name}"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=stem_midi_dir))

    print(f"Downloading merged MIDI: {merged_midi_path}")
    files.download(str(merged_midi_path))


## Optional: Run Inference

In [ ]:
# @title Run Transcription
!python infer.py --audio "{audio_path}"

import os
midi_path = os.path.splitext(audio_path)[0] + ".mid"
if os.path.exists(midi_path):
    print(f"Success! MIDI saved to: {midi_path}")
else:
    print("Error: MIDI file was not generated.")

## Optional: Download Results

In [ ]:
# @title Download MIDI file
if os.path.exists(midi_path):
    files.download(midi_path)
else:
    print("No MIDI file to download.")